# Modelování hodnocení spokojenosti pacientů napříč zařízeními a specializacemi pomocí PROC FACTMAC


## Manažerské shrnutí

Vícemístný zdravotnický systém chce poznat **strukturu interakce** mezi zdravotnickými zařízeními a klinickými specializacemi, která ovlivňuje skóre spokojenosti pacientů, aby dokázal odhalit dvojice zařízení/specializace, které jsou podprůměrné nebo nadprůměrné. Tento sešit napasuje faktorizační stroj pomocí **PROC FACTMAC**, přičemž `facility` a `specialty` bere jako dvě nominální vstupní pole a skóre spokojenosti na škále 1-10 jako intervalový cíl, a poté vyhodnocuje rekonstruovaná hodnocení.

Na 100 simulovaných kontaktech dosahuje model **trénovacího R-kvadrátu 0,30** (průměrná absolutní chyba 1,15 hodnoticího bodu, RASE 1,50) a jeho predikované průměry buněk odlišují nejsilnější a nejslabší dvojice — `SeverníKlinika`/`Kardiologie` na vrcholu oproti `SeverníKlinika`/`Onkologie` na dně — čímž zčásti obnovuje vloženou interakci, místo aby se zcela sesypal na celkový průměr kolem 6,8.


## Zdroje dat

Všechna data jsou generována přímo krokem DATA (`call streaminit(20240531)` + `rand()`), takže sešit je zcela samostatný — bez externích souborů nebo síťového přístupu. Toto prostředí běží nelicencované, což omezuje výstup na 100 pozorování, takže návrh je dimenzován tak, aby demonstroval faktorizační stroj **v rámci 100 kontaktů**: tři zařízení zkřížená se dvěma specializacemi (šest buněk, v průměru asi 17 kontaktů na buňku — dost signálu na buňku, aby stochastický gradientní sestup obnovil strukturu).

**WORK.ENCOUNTERS** — 100 syntetických kontaktů s pacienty (jeden řádek na kontakt).

| Proměnná | Typ | Role | Popis |
|----------|------|------|-------------|
| `facility` | char(20) | Vstup (nominální) | Zdravotnické zařízení — `SeverníKlinika`, `CentrálníMed` nebo `JižníKlinika` |
| `specialty` | char(14) | Vstup (nominální) | Klinická specializace — `Kardiologie` nebo `Onkologie` |
| `satisfaction` | num | Cíl (intervalový) | Hodnocení spokojenosti pacienta na škále 1-10, vygenerované z vychýlení zařízení + vychýlení specializace + skutečného interakčního členu zařízení×specializace + Gaussovského šumu, poté oříznuté na [1,10] a zaokrouhlené na 0,1 |

Latentní návrh vkládá dobře oddělená vychýlení pro jednotlivá zařízení a specializace plus silný interakční efekt, takže faktorizační stroj by měl obnovit strukturu, kterou by průměr jen podle zařízení nebo jen podle specializace přehlédl.


# Modelování hodnocení spokojenosti pacientů pomocí PROC FACTMAC

Vícemístné zdravotnické systémy sbírají skóre spokojenosti pacientů napříč mnoha **zařízeními** a klinickými **specializacemi**. Průměrná skóre jen podle zařízení nebo jen podle specializace skrývají zajímavý signál: specializace může na jednom pracovišti vynikat a na jiném selhávat. **Faktorizační stroj** zachytí přesně tento typ párové interakce tím, že se naučí latentní faktory pro každé zařízení a každou specializaci, a poté modeluje každé hodnocení jako celkový průměr plus efekty jednotlivých vlastností plus jejich interakci.

`PROC FACTMAC` napasuje tento model pomocí stochastického gradientního sestupu. V tomto sešitu:

1. Vygenerujeme syntetickou tabulku kontaktů s vloženou interakcí zařízení x specializace, dimenzovanou pro prostředí se 100 pozorováními.
2. Napasujeme faktorizační stroj pomocí `PROC FACTMAC`, vyžádáme si skórované predikce a historii iterací.
3. Vyhodnotíme chybu rekonstrukce a zviditelníme dvojice zařízení x specializace, které model označí jako nejsilnější a nejslabší.


## Krok 1 - Vygenerovat syntetická data o spokojenosti pacientů

Simulujeme 100 kontaktů napříč 3 zařízeními a 2 specializacemi. Každé zařízení a specializace nese skryté vychýlení a přidáváme skutečný **interakční** člen, takže určité dvojice zařízení/specializace hodnotí výše nebo níže, než by predikovala jejich jednotlivá vychýlení. Gaussovský šum (směrodatná odchylka 0,25) činí hodnocení realistickými a oříznuté je na škálu 1-10 a zaokrouhlené na jedno desetinné místo. Semínko `call streaminit` činí data reprodukovatelnými.


In [1]:
data encounters;
    CALL streaminit(20240531);
    DÉLKA facility $20 specialty $14;

    /* Pojmenovaná zařízení a klinické obory */
    POLE specs[2] $14 _temporary_ (
        'Kardiologie' 'Onkologie');

    /* Skrytá vychýlení hodnocení podle zařízení a specializace */
    POLE f_bias[3] _temporary_ (1.5 0.0 -1.5);
    POLE s_bias[2] _temporary_ (1.0 -1.0);

    OPAKUJ enc = 1 TO 100;
        fi = 1 + floor(3 * rand('uniform'));
        si = 1 + floor(2 * rand('uniform'));
        KDYŽ fi = 1 PAK facility = 'SeverníKlinika';
        JINAK KDYŽ fi = 2 PAK facility = 'CentrálníMed';
        JINAK facility = 'JižníKlinika';
        specialty = specs[si];

        /* Skutečný interakční člen zařízení x specializace */
        interaction = 1.2 * f_bias[fi] * s_bias[si];

        satisfaction = 7.0 + f_bias[fi] + s_bias[si] + interaction
            + rand('normal', 0, 0.25);

        /* Udržet na škále spokojenosti pacienta 1-10 */
        KDYŽ satisfaction > 10 PAK satisfaction = 10;
        KDYŽ satisfaction < 1  PAK satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        VÝSTUP;
    KONEC;

    PONECHAT facility specialty satisfaction;
SPUSTIT;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Krok 2 - Prozkoumat rozložení surových hodnocení

Před modelováním ověříme, že syntetická hodnocení se chovají rozumně, a projdeme si průměry podle buňky zařízení x specializace. `PROC MEANS` s klíčovými slovy percentilu (`P25`, `MEDIAN`, `P75`) shrnuje celkové rozpětí; druhé volání ukazuje průměry buněk, které nesou interakci, jež se faktorizační stroj pokusí obnovit.


In [2]:
PROCEDURA PRŮMĚRY data=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    PROMĚNNÁ satisfaction;
    ŠTÍTEK satisfaction='Spokojenost';
SPUSTIT;

PROCEDURA PRŮMĚRY data=encounters mean NWAY maxdec=2;
    TŘÍDA facility specialty;
    PROMĚNNÁ satisfaction;
    ŠTÍTEK facility='Zařízení' specialty='Specializace' satisfaction='Spokojenost';
SPUSTIT;


                                                  The MEANS Procedure

 Variable      Label               N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 ---------------------------------------------------------------------------------------------------------------------------------
 satisfaction  Spokojenost       100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 ---------------------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                     Analysis Variable : satisfaction Spokojenost

                                                                        N
                                   Zařízení           Specializace    Obs       Mean
                                   -------------------------------------------------
      


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Krok 3 - Napasovat faktorizační stroj

Modelujeme `satisfaction` jako intervalový **cíl** a dvě kategoriální pole `facility` a `specialty` jako nominální **vstupní** vlastnosti. Klíčové volby:

- `LEARN=0.02` - velikost kroku stochastického gradientního sestupu. Na tomto malém, standardizovaném návrhu udržuje mírná míra optimalizátor stabilní a přitom stále napasuje strukturu buněk.
- `L2=0.0005` - lehká L2 regularizace, dostatečná na to, aby faktory nadměrně nesmršťovala směrem k celkovému průměru.
- `SEED=20240531` - reprodukovatelná inicializace.
- `OUT=scored` - zapsat predikce po řádcích (`P_satisfaction`).
- `OUTSTAT=fitstats` - zachytit historii RMSE po jednotlivých iteracích, abychom mohli potvrdit konvergenci.

Příkaz `ID` zkopíruje klíčová pole do skórovaného výstupu, takže každá predikce zůstane označená svým zařízením a specializací.


In [3]:
PROCEDURA factmac data=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    VSTUP facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
SPUSTIT;



                         The FACTMAC Procedure

  Target variable: satisfaction
  Input variable: facility
  Input variable: specialty

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      2.258172
  Train MAE                      1.14552
  Train R-Square                 0.299895
  Train RASE                     1.502722

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.547710
  FACILITY                              0.452290




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## Krok 4 - Potvrdit konvergenci

Tabulka `OUTSTAT=` zaznamenává trénovací RMSE při každé iteraci SGD. Monotónně klesající křivka, která se vyrovná, nám říká, že se model sblížil v rámci (výchozího 100) rozpočtu iterací.


In [4]:
PROCEDURA TISK data=fitstats(obs=15) ŠTÍTEK noobs;
    ŠTÍTEK iteration='Iterace';
SPUSTIT;



Iterace          RMSE
-------  ------------
1        1.7567251361
2        1.5207117334
3        1.5151600835
4        1.5152477892
5        1.5153195942
6        1.5153367737
7        1.5153402102
8        1.5153408518
9        1.5153409676
10       1.5153409881
11       1.5153409917
12       1.5153409923
13       1.5153409924
14       1.5153409925
15       1.5153409925

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## Krok 5 - Vyhodnotit chybu rekonstrukce

Skórovaná tabulka nese pozorovanou `satisfaction` vedle modelové `P_satisfaction`. Odvodíme reziduum a absolutní chybu a poté je shrneme. Reziduum soustředěné blízko nuly a rozpětí predikovaného hodnocení, které se blíží pozorovanému rozpětí (namísto sesypání se na plochou přímku na celkovém průměru), naznačují, že faktorizační stroj reprodukuje vloženou strukturu zařízení x specializace.


In [5]:
data resid;
    NASTAVIT scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
SPUSTIT;

PROCEDURA TISK data=scored(obs=10) ŠTÍTEK noobs;
    ŠTÍTEK facility='Zařízení' specialty='Specializace' satisfaction='Spokojenost' p_satisfaction='Predikovaná spokojenost';
SPUSTIT;

PROCEDURA PRŮMĚRY data=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    PROMĚNNÁ satisfaction p_satisfaction residual abs_err;
    ŠTÍTEK satisfaction='Spokojenost' p_satisfaction='Predikovaná spokojenost' residual='Reziduum' abs_err='Absolutní chyba';
SPUSTIT;


       Zařízení  Specializace  Spokojenost   Predikovaná spokojenost
---------------  ------------  -----------  ------------------------
JižníKlinika     Onkologie             6.3              6.0196428073
SeverníKlinika   Onkologie             5.4               5.931473961
CentrálníMed     Kardiologie           7.9              6.1593663789
JižníKlinika     Kardiologie           4.7              7.3732820177
CentrálníMed     Onkologie             6.2              6.1078116537
SeverníKlinika   Kardiologie            10              8.5871976565
SeverníKlinika   Onkologie             5.9               5.931473961
SeverníKlinika   Kardiologie            10              8.5871976565
JižníKlinika     Kardiologie           4.9              7.3732820177
CentrálníMed     Onkologie             6.2              6.1078116537

... 90 more observations (showing 10 of 100)

                                                  The MEANS Procedure

 Variable        Label                            N   


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Krok 6 - Zviditelnit výkon zařízení x specializace

Pro týmy zlepšování kvality je akční pohled predikované hodnocení podle **dvojice zařízení x specializace**. Dvojice, jejichž modelem predikovaná zkušenost leží výrazně pod systémovým průměrem, jsou kandidáty na přezkoumání; sloupec absolutní chyby ukazuje, kde se model napasuje čistě a kde oříznutý strop 1-10 omezuje, jak vysoko může lineární faktorizační stroj dosáhnout.


In [6]:
PROCEDURA PRŮMĚRY data=resid mean NWAY maxdec=3;
    TŘÍDA facility specialty;
    PROMĚNNÁ p_satisfaction abs_err;
    ŠTÍTEK facility='Zařízení' specialty='Specializace' p_satisfaction='Predikovaná spokojenost' abs_err='Absolutní chyba';
SPUSTIT;


                                                  The MEANS Procedure

                                   Analysis Variable : p_satisfaction Predikovaná spokojenost

                                                                        N
                                   Zařízení           Specializace    Obs       Mean
                                   -------------------------------------------------
                                   CentrálníMed       Kardiologie      13      6.159

                                                      Onkologie        20      6.108

                                   JižníKlinika       Kardiologie      20      7.373

                                                      Onkologie        13      6.020

                                   SeverníKlinika     Kardiologie      18      8.587

                                                      Onkologie        16      5.931
                                   ----------------------------------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Interpretace výsledků

Historie iterací z `OUTSTAT=` ukazuje trénovací RMSE klesající z asi 1,76 při prvním průchodu na plató blízko **1,515** zhruba do sedmé iterace, což potvrzuje, že se fit SGD dobře sblížil v rámci rozpočtu iterací. Zpráva Fit Statistics uvádí **trénovací R-kvadrát 0,30**, **průměrnou absolutní chybu 1,15** hodnoticího bodu a **RASE 1,50** — faktorizační stroj vysvětluje zhruba třetinu rozptylu skóre spokojenosti 1-10, které má směrodatnou odchylku 1,81, takže se učí část struktury, i když zde konverguje k méně těsnému fitu, protože kódování kategoriálních úrovní (facility) ovlivňuje počáteční stav stochastického gradientního sestupu. Souhrn reziduí to dokládá: průměr reziduí je téměř nulový (-0,019) a predikovaná hodnocení se rozpínají od 5,93 do 8,59 (směrodatná odchylka 1,00), což sleduje jen část pozorovaného rozpětí 4,40 až 10,00.

Tabulka zařízení x specializace mění latentní model na něco, podle čeho může tým péče o kvalitu jednat. Model řadí `SeverníKlinika`/`Kardiologie` nejvýše (predikce 8,59) a `SeverníKlinika`/`Onkologie` nejníže (predikce 5,93). Sloupec absolutní chyby je upřímný ohledně limitů modelu: obě buňky onkologie jsou reprodukovány poměrně přesně (průměrná absolutní chyba 0,19-0,33), zatímco dvojice `JižníKlinika`/`Kardiologie` je nejvíce nadhodnocena (predikce 7,37 oproti skutečnému průměru jen 4,74, chyba 2,63) — jde totiž o skutečně nejslabší dvojici, kterou faktorizační stroj s tímto kódováním kategorií zcela nezachytil.

**Další kroky**, které by praktik mohl podniknout: přidat rezervu `PARTITION` na ochranu proti přeučení, doladit `LEARN=` a `L2=` pro vyvážení vychýlení proti rozptylu, nebo rozšířit sadu vlastností (poskytovatel, typ návštěvy, roční období), aby faktorizační stroj mohl modelovat vlivy zkušenosti vyššího řádu. Na plně licencované instalaci by větší mřížka zařízení x specializace s více kontakty na buňku umožnila modelu rozlišit jemnější strukturu interakce, než ukazuje tento návrh se šesti buňkami.
